# Ensemble-Methoden

**Ensemble** = viele schwache Modelle zusammen sind stärker als ein starkes Modell.

**Analogie:** Statt einen einzigen Experten zu befragen, befragst du 100 Menschen — der Durchschnitt ist oft besser als der beste Einzelne (Weisheit der Masse).

## Inhaltsverzeichnis
1. Warum Ensemble-Methoden?
2. Bagging & Pasting
3. Random Forests
4. AdaBoost (Boosting I)
5. Gradient Boosting (Boosting II)
6. XGBoost (Extreme Gradient Boosting)
7. Vergleich aller Methoden


## 1. Warum Ensemble-Methoden?

**Problem:** Ein einzelner Entscheidungsbaum:
- Overfittet stark (lernt Rauschen auswendig)
- Ist instabil (kleine Datenänderung → komplett anderer Baum)

**Idee:** Trainiere viele verschiedene Bäume und kombiniere deren Vorhersagen!

### Zwei grundlegende Ansätze:

**Bagging (Parallel):**
- Trainiere viele Modelle **unabhängig voneinander**
- Kombiniere durch Abstimmung (Klassifikation) oder Mittelwert (Regression)
- Beispiel: Random Forest

**Boosting (Sequentiell):**
- Trainiere Modelle **nacheinander** — jedes fokussiert auf die Fehler des vorherigen
- Stärkere Modelle aus schwachen aufbauen
- Beispiel: AdaBoost, Gradient Boosting, XGBoost


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Boston-Datensatz
dataset = fetch_openml(name='boston', version=1)
X = dataset.data.astype(float)
y = dataset.target.astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 2. Bagging & Pasting

**Bagging** (Bootstrap Aggregating):
- Trainiere n Bäume, jeder auf einem **zufälligen Teilsample MIT Zurücklegen**
- Gleiche Datenpunkte können mehrfach im Sample sein
- Verschiedene Bäume sehen verschiedene "Versionen" der Daten

**Pasting:**
- Wie Bagging, aber **OHNE Zurücklegen**
- Jeder Datenpunkt kommt maximal einmal im Sample vor

**Idee:** Durch den zufälligen Subsampling erzeugen wir "Diversität" — verschiedene Bäume machen verschiedene Fehler, die sich gegenseitig ausgleichen!


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

# Baseline: Ein einzelner Baum
einzelner_baum = DecisionTreeRegressor(random_state=1)
einzelner_baum.fit(X_train, y_train)
print(f"Einzelner Baum:        R² = {einzelner_baum.score(X_test, y_test):.4f}")

# Bagging: 100 schwache Bäume (max_depth=3)
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100,         # 100 Bäume
    max_samples=0.8,          # Jeder Baum sieht 80% der Daten
    bootstrap=True,           # Mit Zurücklegen (Bagging)
    random_state=1
)
bagging.fit(X_train, y_train)
print(f"Bagging (100 Bäume):   R² = {bagging.score(X_test, y_test):.4f}")

# Wie viele Bäume braucht man?
n_trees_list = [1, 5, 10, 20, 50, 100, 200]
scores = []
for n in n_trees_list:
    b = BaggingRegressor(DecisionTreeRegressor(max_depth=3), n_estimators=n, random_state=1)
    b.fit(X_train, y_train)
    scores.append(b.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(n_trees_list, scores, 'o-', color='steelblue')
plt.xlabel("Anzahl Bäume")
plt.ylabel("R²-Score (Test)")
plt.title("Bagging: Mehr Bäume → besser (bis zu einem Punkt)")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Random Forests

**Random Forest = Bagging + Feature-Randomisierung**

Beim normalen Bagging suchen alle Bäume beim Splitten noch unter **allen** Features — das kann dazu führen, dass alle Bäume ähnlich werden (weil sie alle denselben "besten" Split finden).

**Random Forests lösen das:** Bei jedem Split wird nur eine **zufällige Teilmenge** der Features berücksichtigt.

**Effekt:** Noch mehr Diversität zwischen den Bäumen → noch bessere Ensemble-Leistung!

**Wichtige Parameter:**
- `n_estimators`: Anzahl Bäume (mehr = besser, aber langsamer)
- `max_depth`: Tiefe jedes Baums
- `max_features`: Wie viele Features beim Split betrachtet werden (`sqrt` ist Standard)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest trainieren
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,    # Bäume wachsen voll — OK weil Ensemble!
    max_features='sqrt',  # sqrt(n_features) Features pro Split
    random_state=1
)
rf.fit(X_train, y_train)
print(f"Random Forest (100 Bäume): R² = {rf.score(X_test, y_test):.4f}")

# Feature Importance im Random Forest
wichtigkeit = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
wichtigkeit.plot(kind='barh', color='forestgreen')
plt.xlabel("Feature Importance")
plt.title("Random Forest: Welche Merkmale sind wichtig?")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. AdaBoost

**Boosting-Idee:** Jeder Baum fokussiert auf die **Fehler des vorherigen Baums**.

**AdaBoost-Algorithmus:**
1. Trainiere schwachen Klassifikator auf Daten (alle gleich gewichtet)
2. Identifiziere die Datenpunkte, die falsch klassifiziert wurden
3. Erhöhe die Gewichte der falsch klassifizierten Punkte
4. Trainiere nächsten Klassifikator — er muss sich besonders um die schwierigen Punkte kümmern
5. Wiederhole n Mal
6. Finale Vorhersage = gewichtete Abstimmung aller Klassifikatoren

**Analogie:** Ein neuer Schüler lernt genau aus den Fehlern, die der vorherige Schüler gemacht hat.


In [ ]:
from sklearn.ensemble import AdaBoostRegressor

# AdaBoost
ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100,
    learning_rate=0.1,  # Wie stark jeder Baum beiträgt
    random_state=1
)
ada.fit(X_train, y_train)
print(f"AdaBoost (100 Bäume):  R² = {ada.score(X_test, y_test):.4f}")

# Learning Rate Einfluss
for lr in [0.01, 0.1, 0.5, 1.0]:
    ada_lr = AdaBoostRegressor(DecisionTreeRegressor(max_depth=3), 
                                n_estimators=100, learning_rate=lr, random_state=1)
    ada_lr.fit(X_train, y_train)
    print(f"  Learning Rate={lr:.2f}: R² = {ada_lr.score(X_test, y_test):.4f}")

## 5. Gradient Boosting

**Gradient Boosting** ist wie AdaBoost, aber mathematisch eleganter:

Statt die Gewichte der falsch klassifizierten Punkte zu erhöhen, trainiert jeder neue Baum auf den **Residuen** (Fehlern) des vorherigen Modells.

**Algorithmus:**
1. Starte mit einer einfachen Vorhersage (Mittelwert von y)
2. Berechne die Residuen (echte Werte - Vorhersagen)
3. Trainiere neuen Baum auf den **Residuen** (nicht den Originalwerten!)
4. Addiere Vorhersage des neuen Baums (skaliert mit Learning Rate) zum Gesamtmodell
5. Berechne neue Residuen, wiederhole ab Schritt 3

**Analogie:** Wie wenn ein Korrektor die Fehler eines ersten Entwurfs verbessert, dann ein zweiter Korrektor die verbleibenden Fehler, usw.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=1
)
gb.fit(X_train, y_train)
print(f"Gradient Boosting:     R² = {gb.score(X_test, y_test):.4f}")

# Wie entwickelt sich der Fehler mit jedem Baum?
scores_verlauf = []
for y_pred_stage in gb.staged_predict(X_test):
    scores_verlauf.append(r2_score(y_test, y_pred_stage))

plt.figure(figsize=(8, 4))
plt.plot(scores_verlauf, color='darkorange')
plt.xlabel("Anzahl Bäume")
plt.ylabel("R² Score")
plt.title("Gradient Boosting: Score verbessert sich mit jedem Baum")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Beste Anzahl Bäume: {np.argmax(scores_verlauf)+1}")

## 6. XGBoost (Extreme Gradient Boosting)

**XGBoost** ist eine optimierte Implementierung von Gradient Boosting:
- Schneller durch Parallelisierung
- Besser durch regularisierte Bäume
- Behandelt fehlende Werte automatisch
- Gilt als einer der besten Algorithmen für tabellarische Daten

**Geheimt Waffe auf Kaggle-Wettbewerben!**


In [ ]:
try:
    import xgboost as xgb

    xgb_reg = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        random_state=1,
        verbosity=0
    )
    xgb_reg.fit(X_train, y_train)
    print(f"XGBoost:               R² = {xgb_reg.score(X_test, y_test):.4f}")
except ImportError:
    print("XGBoost nicht installiert. Installieren mit: pip install xgboost")
    print("Hier ist ein Platzhalter-Ergebnis: R² ≈ 0.90")

## 7. Großer Vergleich aller Methoden

Lass uns alle Methoden direkt vergleichen!


In [ ]:
from sklearn.linear_model import LinearRegression

# Alle Modelle
alle_modelle = {
    'Lineare Regression (Baseline)': LinearRegression(),
    'Einzelner Baum (kein Limit)': DecisionTreeRegressor(random_state=1),
    'Einzelner Baum (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=1),
    'Bagging (100 Bäume)': BaggingRegressor(DecisionTreeRegressor(max_depth=5), 
                                               n_estimators=100, random_state=1),
    'Random Forest (100 Bäume)': RandomForestRegressor(n_estimators=100, random_state=1),
    'AdaBoost (100 Bäume)': AdaBoostRegressor(DecisionTreeRegressor(max_depth=3),
                                                n_estimators=100, random_state=1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                     learning_rate=0.1, random_state=1),
}

print(f"{'Modell':<40} {'Train R²':>10} {'Test R²':>10}")
print("-" * 63)
for name, modell in alle_modelle.items():
    modell.fit(X_train, y_train)
    train_r2 = modell.score(X_train, y_train)
    test_r2  = modell.score(X_test, y_test)
    diff = train_r2 - test_r2
    hinweis = "⚠️ Overfitting" if diff > 0.15 else "✓"
    print(f"{name:<40} {train_r2:>10.4f} {test_r2:>10.4f}  {hinweis}")

## Zusammenfassung: Wann welche Ensemble-Methode?

| Methode | Stärken | Schwächen | Wann verwenden? |
|---------|---------|-----------|----------------|
| **Bagging** | Reduziert Varianz, einfach | Wenig Bias-Reduktion | Als einfaches Ensemble |
| **Random Forest** | Robust, Feature Importance, kaum Tuning nötig | Weniger interpretierbar | Standard-Wahl für viele Probleme! |
| **AdaBoost** | Gute Performance | Empfindlich für Ausreißer | Wenn du Boosting ohne viel Tuning willst |
| **Gradient Boosting** | Sehr gute Performance | Langsam, viel Tuning | Wenn maximale Genauigkeit wichtig ist |
| **XGBoost** | Sehr schnell, beste Performance | Komplexes Tuning | Wettbewerbe, Produktionssysteme |

### Ensemble-Methoden vs. einzelne Modelle

**Wann lohnt es sich?**
- Wenn maximale Genauigkeit wichtig ist (Produktion, Wettbewerbe)
- Wenn du genug Daten hast
- Wenn Interpretierbarkeit keine Priorität ist

**Wann lieber ein einzelnes Modell?**
- Wenn Interpretierbarkeit wichtig ist (Arzt, Richter muss Entscheidung erklären)
- Bei sehr kleinen Datensätzen
- Wenn Geschwindigkeit (Training/Inferenz) kritisch ist
